In [1]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
import re

import imageio_ffmpeg
import os

os.environ["IMAGEIO_FFMPEG_EXE"] = imageio_ffmpeg.get_ffmpeg_exe()


In [3]:
!pip install num2words

In [4]:
import num2words
import torch
from transformers import AutoModelForImageTextToText


# model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cuda") # Changed from "cuda" to "cpu" to run on CPU

model.eval()

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(729, 1152)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-26): 27 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_features=1152, 

In [5]:
# THIS HAS TO CHANGE - HEAVY EXPERIMENTATION
# PROMPT_EVENT_ONLY = """
#  Describe all the key events in the video.
#  Ignore
#  Prefer specific actions, scenes, and in-text description rather than general constatations.
#  One sentence for one event.
# """

# PROMPT_EVENT_ONLY = """
# Describe the most important observable event in this video segment in one concise sentence.

# Prioritize dynamic actions, interactions, demonstrations, or notable visual changes over static or repetitive scenes.

# Avoid generic descriptions of people merely standing, presenting, or facing the camera unless a meaningful action occurs.

# Only describe directly observable information.
# """


# PROMPT_EVENT_ONLY = """
# Describe the important events in this video segment in one concise sentence.
# Focus on very concise events and ignore everyday actions.
# Only describe directly observable information.
# """

PROMPT_EVENT_ONLY = """
Describe all the key events in the video.
Do not repeat events.
"""



# PROMPT_EVENT_ONLY = """
# Describe the main events in this video segment in 2-3 sentneces.
# Focus only on very concise events.
# One sentnece per event.
# """


# PROMPT_EVENT_ONLY = """
# Describe the main event in this video segment in exactly one concise sentence.

# Focus on actions that contribute to understanding what is happening in the video.

# Omit background and prioritise eye-catching events.

# Only describe information directly observable in the frames.
# """


# Ignore background scences and unecessary details (e.g. colour).


# PROMPT_EVENT_ONLY = """
#  Describe all the key events in the video.

#   One sentence for one event.
#   Don't repeat similar events.
#   Ignore colours and background details.
# """



# PROMPT_EVENT_ONLY = """
# Describe the clearest visible physical action or visual event in this video segment in one concise sentence.

# Only describe information directly visible in the frames.

# Do not mention talking, speaking, explaining, or discussion unless clearly visible and unambiguous.

# Prefer specific physical actions, object interactions, or visible text over general scene descriptions.

# Do not assume emotions, intentions, or off-screen events.
# """

# average ().
# PROMPT_EVENT_ONLY = """
# Describe the main event occuring in this video segment with one single sentence.

# Only mention the actions that are observable from the frames.

# Prefer a short, specific event description rather than a long or general description.


# Do not assume things like speech.

# """


# Describe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved.
# """

#Jesse:
#1. "Describe the single most important event in this video segment in one concise sentence. Focus on what is happening and who is involved.
#gave very repetitive answers, and often missed important events. Mainly has "man in white shirt speaking to camera...".
#Could be seen as most "important" as it is visible the largest part of the video segments
#2. To get rid of this repitive stuff I tried to incorporate the events that are already outputted by the VLM in the prompt
#to give it a bit of context from what already has happened.
# Addition to 1: Prevent describing events that are already outputted, which are: + all_outputs descriptions so far.
#This did not help at all, only made it worse. Less focus on describing right events and even more repitive events.
#Have a feeling this extra constraining is not working at all and we have to be clear in 1/2 sentence and being careful with the words we choose
#3. Describe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved.
#Really finds more distinctive events, does not keep describing the man talking to the camera


#Nikitas:
#"Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"

#Alex
#1) Retrieve all salient events in the video. Shit for both video 5 and video 6!
#2) Describe all the key events in the video. Much better for video 5 and incredibly better for video 6. A lot of yapping though and overlap.
#3) "Describe all the key events in the video." "Return only events that are important for understanding the video."
# A bit better, but still hasn't captured salient events.

In [6]:
# this function runs the model per input chunk
# need to define number of frames used per chunk + number of output tokens
def run_vlm_on_video(video_path, prompt, num_frames=8, max_new_tokens=500):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]


    inputs = processor.apply_chat_template(
    messages,
    num_frames = num_frames,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make


    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()


In [7]:
def split_video_into_chunks(video_path, output_dir, num_chunks=10):

    os.makedirs(output_dir, exist_ok=True)

    # Reuse existing chunks if available
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))

    if existing_chunks:

        chunk_paths = []

        for chunk_path in existing_chunks:

            filename = os.path.basename(chunk_path)

            parts = (
                filename
                .replace("chunk_", "")
                .replace(".mp4", "")
                .split("_")
            )

            start = float(parts[0])
            end = float(parts[1])

            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")

        return chunk_paths

    # Load video
    video = VideoFileClip(video_path)

    duration = video.duration

    # Compute equal chunk duration
    chunk_duration = duration / num_chunks

    chunk_paths = []

    for i in range(num_chunks):

        start = i * chunk_duration

        # ensure final chunk reaches exact end
        if i == num_chunks - 1:
            end = duration
        else:
            end = (i + 1) * chunk_duration



        chunk_path = os.path.join(
            output_dir,
        f"chunk_{start:07.2f}_{end:07.2f}.mp4"
        )


        # compatibility with moviepy versions
        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()

        chunk_paths.append((chunk_path, start, end))

    video.close()

    return chunk_paths

# # splitting the entire video into chunks of lenght x seconds


# def split_video_into_chunks(video_path, output_dir, chunk_length=8):
#     os.makedirs(output_dir, exist_ok=True)

#     # If chunks already exist, reuse them
#     existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))
#     if existing_chunks:
#         chunk_paths = []

#         for chunk_path in existing_chunks:
#             filename = os.path.basename(chunk_path)
#             parts = filename.replace("chunk_", "").replace(".mp4", "").split("_")
#             start = int(parts[0])
#             end = int(parts[1])
#             chunk_paths.append((chunk_path, start, end))

#         print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")
#         return chunk_paths

#     # Otherwise, create chunks
#     video = VideoFileClip(video_path)
#     # duration = int(video.duration)
#     duration = video.duration

#     chunk_paths = []

#     for start in range(0, int(duration), chunk_length): # change to int(duration) from duration
#         end = min(start + chunk_length, int(duration)) # change to int(duration)

#         chunk_path = os.path.join(
#             output_dir,
#             f"chunk_{start:04d}_{end:04d}.mp4"
#         )

#         if hasattr(video, "subclipped"):
#             clip = video.subclipped(start, end)
#         else:
#             clip = video.subclip(start, end)

#         clip.write_videofile(
#             chunk_path,
#             codec="libx264",
#             audio=False,
#             logger=None
#         )

#         clip.close()
#         chunk_paths.append((chunk_path, start, end))

#     video.close()
#     return chunk_paths

In [8]:

# CHOOSE VIDEO
video_number = 5
video_path = f"video_{video_number}.mp4"


video = VideoFileClip(video_path)
    # duration = int(video.duration)
duration = video.duration


num_chunks = 1
chunk_length = int(duration / num_chunks)



frames_per_second = 2
# num_frames = round(chunk_length*frames_per_second)
num_frames = 32
max_new_tokens = 800


# PARAMETERS
# chunk length is the lenght in seconds of each chunk
# num frames is the number of frames per chunk
# max tokens is the upper bound on number of words per generated output for each chunk

# chunk_length = 10
# num_frames = 20
# max_new_tokens = 128

output_dir = f"video_{video_number}_{num_chunks}_chunks"

chunks = split_video_into_chunks(
    video_path,
    output_dir=output_dir,
    num_chunks= num_chunks# changed to 8 seconds
)

all_outputs = []

for chunk_path, start, end in chunks:

    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        num_frames=num_frames,
        max_new_tokens=max_new_tokens # match it to length of 1-2 sentences
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)


# output = run_vlm_on_video(
#     video_path,
#     PROMPT_EVENT_ONLY,
#     num_frames=64,
#     max_new_tokens=800 # match it to length of 1-2 sentences
# )

# # print(f"\nCHUNK {start}-{end} seconds")
# print(output)

Reusing 1 existing chunks from video_5_1_chunks


You have used fast image processor with LANCZOS resample which not yet supported for torch.Tensor. BICUBIC resample will be used as an alternative. Please fall back to image processor if you want full consistency with the original model.



CHUNK 0.0-111.02 seconds
The video begins with a title screen displaying "GMC Certified Service" and "Replacing Your Tires." It transitions to a man in a white shirt with the GMC logo standing in front of a dark blue background with a GMC truck. The man, identified as Dave Campbell, provides information about GMC certified service. The scene then shifts to a man in a black shirt working on a vehicle, with the text "GMC Certified Service" and "Dave Campbell" appearing on the screen. The video continues with a close-up of a tire being rotated, followed by a comparison of good and bad tire treads. The video emphasizes the importance of rotating tires every 7,500 miles. The final segment shows a man in a white shirt standing in front of a dark blue background with a GMC truck, with the text "GMC Certified Service" and "gmccertifiedservice.com" appearing on the screen. The video concludes with a title screen displaying "GMC Certified Service" and "Replacing Your Tires."


In [ ]:
# import pandas as pd
# import re

# # Split the 'output' string into sentences
# # This regex attempts to split sentences while handling common abbreviations (e.g., Mr., Mrs.)
# sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s+', output)

# # Filter out any empty strings that might result from splitting
# sentences = [s.strip() for s in sentences if s.strip()]

# # Create the DataFrame with each sentence as a row
# df_predicted_events = pd.DataFrame(sentences, columns=['Predicted Event'])

# # Save the DataFrame to CSV
# df_predicted_events.to_csv('results_32frames.csv', index = False)

In [9]:
# def parse_vlm_events(text):

#     parsed_events = []

#     # split on sentence-ending punctuation
#     events = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())

#     for event in events:

#         # clean whitespace and punctuation
#         cleaned_event = event.strip(" ,.-\n\t\"'")

#         # only keep non-empty events
#         if cleaned_event:

#             parsed_events.append({
#                 "event": cleaned_event,
#                 "start_time": None,
#                 "end_time": None
#             })

#     return parsed_events




def parse_vlm_events(text):

    parsed_events = []

    # remove quotes globally first
    text = text.replace('"', '').strip()  # this was done due to wierd otuput

    # split on sentence-ending punctuation
    events = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

    for event in events:

        # clean whitespace and punctuation
        cleaned_event = event.strip(" ,.-\n\t'")

        # only keep non-empty events
        if cleaned_event:

            parsed_events.append({
                "event": cleaned_event,
                "start_time": None,
                "end_time": None
            })

    return parsed_events



# def parse_vlm_events(text):
#     # Expects the output per chunk (text is for chunk output) to be normal text descritpion
#     # Should work also if each chunk gives multiple events

#     parsed_events = []
#     parsed_events.append({
#             "event": text.strip(" ,.-"),   # remove comma, dot etc.
#             "start_time": None,
#             "end_time": None
#     })

#     return parsed_events

In [10]:
# go over all chunk outputs in chronological order
# and store a list of salient event descriptions
predicted_events = []

for item in all_outputs:
    parsed = parse_vlm_events(item["vlm_output"])
    predicted_events.extend([e["event"] for e in parsed])


In [11]:
print(predicted_events)

['The video begins with a title screen displaying GMC Certified Service and Replacing Your Tires', 'It transitions to a man in a white shirt with the GMC logo standing in front of a dark blue background with a GMC truck', 'The man, identified as Dave Campbell, provides information about GMC certified service', 'The scene then shifts to a man in a black shirt working on a vehicle, with the text GMC Certified Service and Dave Campbell appearing on the screen', 'The video continues with a close-up of a tire being rotated, followed by a comparison of good and bad tire treads', 'The video emphasizes the importance of rotating tires every 7,500 miles', 'The final segment shows a man in a white shirt standing in front of a dark blue background with a GMC truck, with the text GMC Certified Service and gmccertifiedservice.com appearing on the screen', 'The video concludes with a title screen displaying GMC Certified Service and Replacing Your Tires']


In [12]:
#Get our annotations
ANNOTATIONS_PATH = "Video5_Nikitas_WithoutSound.txt"

video_annotations = []

#Read annotation file
with open(ANNOTATIONS_PATH, "r") as f:
    for line in f:
        #Split on comma and take the first element
        description = line.strip().split(",")[0]
        if description:

          video_annotations.append(description)
          print(description)

Branding start screen of GMC's tire replacement service
Person talking in front of a GMC car
Technician working on a car tire
Comparison of a tire with a good and bad tread
Displayed text recommending tire rotation every 7500 miles
Demonstration of checking tire tread depth using a tool and a coin
Tire spinning while mounted
People interacting near a car
Branding end screen of GMC's tire replacement service


In [13]:
print(len(video_annotations))

9


In [14]:

model_transformer = SentenceTransformer('all-MiniLM-L6-v2')

# embed ground truth and predicted event sentences
gt_embeddings = model_transformer.encode(video_annotations)
pred_embeddings = model_transformer.encode(predicted_events)


# computing pairwise cosine similarity between ground truth and predicted embeddings
sim_matrix = cosine_similarity(gt_embeddings, pred_embeddings) # maybe other way around

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
print(sim_matrix)

[[ 0.7078219   0.51725215  0.38732442  0.40256572  0.28302133  0.24871358
   0.50327873  0.6843119 ]
 [ 0.43534628  0.5287634   0.35440066  0.5011145   0.15354419  0.12699819
   0.55256814  0.43012613]
 [ 0.44390246  0.13842341  0.23641184  0.3848899   0.42992812  0.37305862
   0.19544482  0.43742833]
 [ 0.23130754 -0.009624    0.00289955 -0.01281048  0.6706326   0.4130995
   0.00714076  0.23925947]
 [ 0.42768842  0.12681876  0.09439218  0.22298217  0.49073756  0.65555453
   0.20097685  0.41235536]
 [ 0.30081758  0.0686063   0.09937751  0.09780814  0.5080099   0.39833117
   0.12260838  0.30695644]
 [ 0.29265144 -0.0081921  -0.01527098  0.0325311   0.54390347  0.48150402
  -0.01643757  0.2916875 ]
 [ 0.22118673  0.2843902   0.06354327  0.3429392   0.12075163  0.15590352
   0.26527545  0.20012532]
 [ 0.70275813  0.5017122   0.40717083  0.39978713  0.32048005  0.23843738
   0.5269612   0.7119421 ]]


In [16]:
print(sim_matrix.shape)

(9, 8)


In [17]:
print(np.mean((sim_matrix)))

0.31386533


In [18]:
# THRESHOLD IS SOMETHING THAT CAN BE CHANGED (0.5 for now)


threshold = 0.5

matches = []
used_preds = set()

for i, gt in enumerate(video_annotations):

    best_j = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_j]

    if best_score >= threshold and best_j not in used_preds:
        matches.append((i, gt, predicted_events[best_j], best_score))
        used_preds.add(best_j)

In [19]:

# NUMERICAL RESULTS (check if needed)

TP = len(matches)
FN = len(video_annotations) - TP
FP = len(predicted_events) - TP

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)\




In [20]:
print(len(matches))

5


In [21]:
print(TP)
print(FN)
print(FP)

5
4
3


In [22]:
print(precision)
print(recall)
print(f1)

0.625
0.5555555555555556
0.5882352941176471


In [23]:
print(len(video_annotations))

9


In [24]:
# FINDING THE MISSED EVENTS (i.e if for a ground truth event no predicted was found to be similar)

matched_gt_indices = set([m[0] for m in matches])

missed_events = [
    (i, gt)
    for i, gt in enumerate(video_annotations)
    if i not in matched_gt_indices
]

print("\nMissed events:")

for i, e in missed_events:
    print(f"- Event {i}: {e}")


Missed events:
- Event 2: Technician working on a car tire
- Event 5: Demonstration of checking tire tread depth using a tool and a coin
- Event 6: Tire spinning while mounted
- Event 7: People interacting near a car


In [25]:
print("\nNon-missed events:")

for i, gt, pred, score in matches:
    print(f"- Event {i}: {gt} (matched with '{pred}' with score: {score:.3f})")


Non-missed events:
- Event 0: Branding start screen of GMC's tire replacement service (matched with 'The video begins with a title screen displaying GMC Certified Service and Replacing Your Tires' with score: 0.708)
- Event 1: Person talking in front of a GMC car (matched with 'The final segment shows a man in a white shirt standing in front of a dark blue background with a GMC truck, with the text GMC Certified Service and gmccertifiedservice.com appearing on the screen' with score: 0.553)
- Event 3: Comparison of a tire with a good and bad tread (matched with 'The video continues with a close-up of a tire being rotated, followed by a comparison of good and bad tire treads' with score: 0.671)
- Event 4: Displayed text recommending tire rotation every 7500 miles (matched with 'The video emphasizes the importance of rotating tires every 7,500 miles' with score: 0.656)
- Event 8: Branding end screen of GMC's tire replacement service (matched with 'The video concludes with a title screen

In [26]:
print(f"Model: {model_path}\n")

print(f"Video {video_number}")
print("Prompt given: ", PROMPT_EVENT_ONLY)

print(f"Chunk Length: {chunk_length}")
print(f"Max Tokens used (per Chunk) : {max_new_tokens}")
print(f"Number of Frames (per Chunk) :{num_frames}")

print(f"Threshold: {threshold}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}\n")

print("===\n")

for i in range(len(predicted_events)):
    print(f"Predicted event {i}:{predicted_events[i]}")






Model: HuggingFaceTB/SmolVLM2-2.2B-Instruct

Video 5
Prompt given:  
Describe all the key events in the video.
Do not repeat events.

Chunk Length: 111
Max Tokens used (per Chunk) : 800
Number of Frames (per Chunk) :32
Threshold: 0.5
Precision: 0.625
Recall:    0.556
F1-score:  0.588

===

Predicted event 0:The video begins with a title screen displaying GMC Certified Service and Replacing Your Tires
Predicted event 1:It transitions to a man in a white shirt with the GMC logo standing in front of a dark blue background with a GMC truck
Predicted event 2:The man, identified as Dave Campbell, provides information about GMC certified service
Predicted event 3:The scene then shifts to a man in a black shirt working on a vehicle, with the text GMC Certified Service and Dave Campbell appearing on the screen
Predicted event 4:The video continues with a close-up of a tire being rotated, followed by a comparison of good and bad tire treads
Predicted event 5:The video emphasizes the importance o

In [ ]:
text_file = []

text_file.append(f"Model: {model_path}")
text_file.append(f"Video: {video_number}")
text_file.append(f"Prompt given:\n{PROMPT_EVENT_ONLY}")
text_file.append(f"Chunk Length: {chunk_length}")
text_file.append(f"Max Tokens used (per Chunk): {max_new_tokens}")
text_file.append(f"Number of Chunks: {num_chunks}")
text_file.append(f"Number of Frames (per Chunk): {num_frames}")
text_file.append(f"Threshold: {threshold}")

text_file.append("\n=== METRICS ===")
text_file.append(f"Precision: {precision:.3f}")
text_file.append(f"Recall:    {recall:.3f}")
text_file.append(f"F1-score:  {f1:.3f}")
text_file.append(f"Hallucinated events number: {FP}")
text_file.append(f"Correctly predicted events number: {TP}")
text_file.append(f"Missed out events number: {FN}")



text_file.append("\n=== PREDICTED EVENTS ===")

for i, event in enumerate(predicted_events):
    text_file.append(f"Predicted event {i}: {event}")


# MISSED EVENTS
matched_gt_indices = set([m[0] for m in matches])

missed_events = [
    (i, gt)
    for i, gt in enumerate(video_annotations)
    if i not in matched_gt_indices
]

text_file.append("\n=== MISSED EVENTS ===")

for i, e in missed_events:
    text_file.append(f"Missed event {i}: {e}")


# NON-MISSED EVENTS
text_file.append("\n=== MATCHED EVENTS ===")

for i, gt, pred, score in matches:
    text_file.append(
        f"Matched GT event {i}: {gt}"
    )
    text_file.append(
        f"    Predicted: {pred}"
    )
    text_file.append(
        f"    Similarity score: {score:.3f}"
    )


# SAVE FILE
os.makedirs("vlm_outputs", exist_ok=True)

output_path = f"vlm_outputs/{video_number}_results_step3.txt"

with open(output_path, "a") as f:
    f.write("\n".join(text_file))

print(f"Results saved to: {output_path}")

Results saved to: vlm_outputs/5_name_prompt_indicator.txt


## ADD STEP 4